## Part 1 - Setup and SAM-Med2D Fine-tuning

In [ ]:
import os
BASE = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
%cd $BASE
!git clone https://github.com/OpenGVLab/SAM-Med2D/
%cd $BASE/SAM-Med2D
!pip install -e . -q

/kaggle/working
Cloning into 'SAM-Med2D'...
remote: Enumerating objects: 350, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 350 (delta 107), reused 47 (delta 47), pack-reused 205 (from 1)
Receiving objects: 100% (350/350), 30.47 MiB | 36.97 MiB/s, done.
Resolving deltas: 100% (162/162), done.
/kaggle/working/SAM-Med2D
ERROR: file:///kaggle/working/SAM-Med2D does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [ ]:
# Create the checkpoint directory in SAM-Med2D
!mkdir -p $BASE/SAM-Med2D/checkpoint

# Download the pretrained SAM-Med2D weights used for fine-tuning.
PRETRAINED_SAM_ID = "1ARiB5RkSsWmAB_8mqWnwDF8ZKTtFwsjl"
PRETRAINED_SAM_PATH = f"{BASE}/SAM-Med2D/checkpoint/sam_med2d.pth"
if not os.path.exists(PRETRAINED_SAM_PATH):
    !gdown --id {PRETRAINED_SAM_ID} -O {PRETRAINED_SAM_PATH}
assert os.path.exists(PRETRAINED_SAM_PATH), (
    f"❌ Missing pretrained SAM-Med2D weights at {PRETRAINED_SAM_PATH}"
)
!ls -lh $BASE/SAM-Med2D/checkpoint


/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1ARiB5RkSsWmAB_8mqWnwDF8ZKTtFwsjl
From (redirected): https://drive.google.com/uc?id=1ARiB5RkSsWmAB_8mqWnwDF8ZKTtFwsjl&confirm=t&uuid=907fe94c-cc65-43c3-8436-9b4b8debfb58
To: /kaggle/working/SAM-Med2D/checkpoint/sam_med2d.pth
100%|██████████████████████████████████████| 2.56G/2.56G [00:29<00:00, 85.9MB/s]
total 2.4G
-rw-r--r-- 1 root root 2.4G Aug 18  2023 sam_med2d.pth


In [ ]:
# =========================================================
# DOWNLOAD DATASET ZIP FROM GOOGLE DRIVE
# =========================================================

!gdown --id 1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx  # dataset_BTXRD (one image, one mask): shared with PGA

!unzip -oq dataset_BTXRD.zip

# Remove the previous directory if it exists
!rm -rf dataset_BTXRD
# Then extract the archive
!unzip -oq dataset_BTXRD.zip

/usr/local/lib/python3.12/dist-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx
From (redirected): https://drive.google.com/uc?id=1Ep3w8GujpCQ5Qb6nlXKxXhsCxlc3k8gx&confirm=t&uuid=d8c020b8-f2f7-4881-a1bb-eaa891faf50a
To: /kaggle/working/SAM-Med2D/dataset_BTXRD.zip
100%|███████████████████████████████████████| 1.71G/1.71G [00:14<00:00, 119MB/s]


In [ ]:
# ── Conversion: JSON annotations -> per-polygon PNG masks ──────────────────────
# Each polygon (lesion) -> one dedicated mask file (IMG001768_1.png, IMG001768_2.png...)
# After this step: train=1859 samples, val=211, test=248 (exactly matched to PGA)

import cv2, json, numpy as np, os, glob

%cd $BASE/SAM-Med2D

def build_per_polygon_masks(split):
    ann_dir  = f"dataset_BTXRD/{split}/annotations"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"

    if not os.path.exists(ann_dir):
        print(f"[!] {split}: annotations directory not found, skipped."); return

    # Remove all previous masks (merged one-per-image) to avoid incorrect counting
    old = glob.glob(f"{mask_dir}/*.png")
    for p in old: os.remove(p)
    print(f"[*] {split}: removed {len(old)} legacy masks")

    n_written = 0
    for json_path in sorted(glob.glob(f"{ann_dir}/*.json")):
        base = os.path.splitext(os.path.basename(json_path))[0]

        # Locate the source image file (.png / .jpg)
        img_path = None
        for ext in ['.png', '.jpg', '.jpeg']:
            p = os.path.join(img_dir, base + ext)
            if os.path.exists(p): img_path = p; break
        if img_path is None:
            print(f"  [!] Image not found for {base}"); continue

        img  = cv2.imread(img_path)
        H, W = img.shape[:2]
        data = json.load(open(json_path))

        polygons = [s for s in data.get('shapes', [])
                    if s.get('shape_type') == 'polygon']

        for i, shape in enumerate(polygons, start=1):
            pts  = np.array(shape['points'], dtype=np.float32)
            pts  = pts.reshape((-1, 1, 2)).astype(np.int32)
            mask = np.zeros((H, W), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 255)
            out  = os.path.join(mask_dir, f"{base}_{i}.png")
            cv2.imwrite(out, mask)
            n_written += 1

    print(f"[✅] {split}: created {n_written} per-polygon mask files")

for split in ['train', 'val', 'test']:
    build_per_polygon_masks(split)

print("\n✅ Done. Updated sample counts:")
for split in ['train', 'val', 'test']:
    n = len(glob.glob(f"dataset_BTXRD/{split}/masks/*.png"))
    print(f"  {split}: {n} mask files")


/kaggle/working/SAM-Med2D
[*] train: removed 1493 legacy masks
[✅] train: created 1847 per-polygon mask files
[*] val: removed 187 legacy masks
[✅] val: created 239 per-polygon mask files
[*] test: removed 187 legacy masks
[✅] test: created 232 per-polygon mask files

✅ Done. Updated sample counts:
  train: 1847 mask files
  val: 239 mask files
  test: 232 mask files


In [ ]:
%cd $BASE/SAM-Med2D
import os, json

def create_mapping_json(split):
    img_dir = f"dataset_BTXRD/{split}/images"
    mask_dir = f"dataset_BTXRD/{split}/masks"
    image2label = {}

    if not os.path.exists(img_dir) or not os.path.exists(mask_dir):
        print(f"[!] Directory not found for split {split}")
        return

    all_masks = os.listdir(mask_dir) if os.path.exists(mask_dir) else []

    for img_name in os.listdir(img_dir):
        if not img_name.endswith(('.png', '.jpg', '.jpeg')): continue
        base = os.path.splitext(img_name)[0]
        img_path = os.path.abspath(os.path.join(img_dir, img_name))
        matched_masks = [os.path.abspath(os.path.join(mask_dir, m))
                         for m in all_masks
                         if os.path.splitext(m)[0] == base
                         or os.path.splitext(m)[0].startswith(f"{base}_")]
        if matched_masks:
            image2label[img_path] = sorted(matched_masks)

    json_out = f"dataset_BTXRD/image2label_{split}.json"
    with open(json_out, 'w', encoding='utf-8') as f:
        json.dump(image2label, f, indent=4)
    print(f"[*] Created mapping for {split}: {len(image2label)} images -> {json_out}")

create_mapping_json("train")
create_mapping_json("val")

# Create label2image_val.json (required by val_one_epoch in train.py)
def create_evaluation_json(split="test"):
    mask_dir = f"dataset_BTXRD/{split}/masks"
    img_dir  = f"dataset_BTXRD/{split}/images"
    label2image = {}

    if not os.path.exists(mask_dir) or not os.path.exists(img_dir):
        print(f"[!] Directory not found for split: '{split}'"); return

    img_dict = {}
    for f in os.listdir(img_dir):
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_dict[os.path.splitext(f)[0]] = f

    for mask_name in os.listdir(mask_dir):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')): continue
        mask_base = os.path.splitext(mask_name)[0]
        img_base  = mask_base.split("_")[0] if "_" in mask_base else mask_base
        if img_base in img_dict:
            label2image[f"dataset_BTXRD/{split}/masks/{mask_name}"] =                         f"dataset_BTXRD/{split}/images/{img_dict[img_base]}"

    out = f"dataset_BTXRD/label2image_{split}.json"
    with open(out, "w", encoding="utf-8") as f_out:
        json.dump(label2image, f_out, indent=4, ensure_ascii=False)
    print(f"[*] label2image_{split}.json: {len(label2image)} samples -> {out}")

create_evaluation_json("test")
create_evaluation_json("val")


/kaggle/working/SAM-Med2D
[*] Created mapping for train: 1493 images -> dataset_BTXRD/image2label_train.json
[*] Created mapping for val: 187 images -> dataset_BTXRD/image2label_val.json
[*] label2image_test.json: 232 samples -> dataset_BTXRD/label2image_test.json
[*] label2image_val.json: 239 samples -> dataset_BTXRD/label2image_val.json


In [ ]:
%%writefile $BASE/SAM-Med2D/train.py
# train.py: training now uses the PGA covering-prompt protocol too (50/50
# zoom-out expansion ratio 0.15-0.45 or off-center shift ratio 0.30 per sample),
# replacing the original authors' get_boxes_from_mask box-noise. Unlike PGA-UNet
# itself, which trains zoom-out only, SAM-Med2D trains on both modes here.
# Validation during training and test.py continue to use the PGA covering-prompt protocol.
# Fix 1: make the Apex import optional (Apex is not required on Colab)
# Fix 2: randint crash when iter_point < 3

from segment_anything import sam_model_registry, SamPredictor
import torch.nn as nn
import torch
import argparse
import os
from torch import optim
from torch.utils.data import DataLoader
from DataLoader import TrainingDataset, TestingDataset, stack_dict_batched
from torch.nn import functional as F
from utils import FocalDiceloss_IoULoss, get_logger, generate_point, setting_prompt_none
from metrics import SegMetrics
import time
from tqdm import tqdm
import numpy as np
import datetime# FIX 1: Apex is optional: only needed when using --use_amp
try:
    from apex import amp
    APEX_AVAILABLE = True
except ImportError:
    APEX_AVAILABLE = False
import random


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--work_dir", type=str, default="workdir", help="work dir")
    parser.add_argument("--run_name", type=str, default="sam-med2d", help="run model name")
    parser.add_argument("--epochs", type=int, default=50, help="number of epochs")
    parser.add_argument("--early_stop", type=int, default=15, help="early stopping patience")
    parser.add_argument("--batch_size", type=int, default=2, help="train batch size")
    parser.add_argument("--image_size", type=int, default=256, help="image_size")
    parser.add_argument("--mask_num", type=int, default=5, help="get mask number")
    parser.add_argument("--data_path", type=str, default="data_demo", help="train data path")
    parser.add_argument("--val_data_path", type=str, default=None, help="val data path (default same as data_path)")
    parser.add_argument("--metrics", nargs='+', default=['iou', 'dice'], help="metrics")
    parser.add_argument('--device', type=str, default='cuda')
    parser.add_argument("--lr", type=float, default=1e-5, help="learning rate")
    parser.add_argument("--resume", type=str, default=None, help="load resume")
    parser.add_argument("--model_type", type=str, default="vit_b", help="sam model_type")
    parser.add_argument("--sam_checkpoint", type=str, default="pretrain_model/sam-med2d_b.pth", help="sam checkpoint")
    parser.add_argument("--iter_point", type=int, default=8, help="point iterations")
    parser.add_argument('--lr_scheduler', type=str, default=None, help='lr scheduler')
    parser.add_argument("--point_list", type=list, default=[1, 3, 5, 9], help="point_list")
    parser.add_argument("--multimask", type=bool, default=True, help="ouput multimask")
    parser.add_argument("--encoder_adapter", type=bool, default=True, help="use adapter")
    parser.add_argument("--use_amp", type=bool, default=False, help="use amp")
    args = parser.parse_args()
    if args.resume is not None:
        args.sam_checkpoint = None
    return args


def to_device(batch_input, device):
    device_input = {}
    for key, value in batch_input.items():
        if value is not None:
            if key=='image' or key=='label':
                device_input[key] = value.float().to(device)
            elif type(value) is list or type(value) is torch.Size:
                 device_input[key] = value
            else:
                device_input[key] = value.to(device)
        else:
            device_input[key] = value
    return device_input


def prompt_and_decoder(args, batched_input, model, image_embeddings, decoder_iter=False):
    if batched_input["point_coords"] is not None:
        points = (batched_input["point_coords"], batched_input["point_labels"])
    else:
        points = None

    if decoder_iter:
        with torch.no_grad():
            sparse_embeddings, dense_embeddings = model.prompt_encoder(
                points=points,
                boxes=batched_input.get("boxes", None),
                masks=batched_input.get("mask_inputs", None),
            )
    else:
        sparse_embeddings, dense_embeddings = model.prompt_encoder(
            points=points,
            boxes=batched_input.get("boxes", None),
            masks=batched_input.get("mask_inputs", None),
        )

    low_res_masks, iou_predictions = model.mask_decoder(
        image_embeddings=image_embeddings,
        image_pe=model.prompt_encoder.get_dense_pe(),
        sparse_prompt_embeddings=sparse_embeddings,
        dense_prompt_embeddings=dense_embeddings,
        multimask_output=args.multimask,
    )

    if args.multimask:
        max_values, max_indexs = torch.max(iou_predictions, dim=1)
        max_values = max_values.unsqueeze(1)
        iou_predictions = max_values
        low_res = []
        for i, idx in enumerate(max_indexs):
            low_res.append(low_res_masks[i:i+1, idx])
        low_res_masks = torch.stack(low_res, 0)

    masks = F.interpolate(low_res_masks, (args.image_size, args.image_size),
                          mode="bilinear", align_corners=False)
    return masks, low_res_masks, iou_predictions


def train_one_epoch(args, model, optimizer, train_loader, epoch, criterion):
    train_loader = tqdm(train_loader)
    train_losses = []
    train_iter_metrics = [0] * len(args.metrics)

    for batch, batched_input in enumerate(train_loader):
        batched_input = stack_dict_batched(batched_input)
        batched_input = to_device(batched_input, args.device)

        # This project is box-only: unlike the original authors' random box/point
        # choice, always drop the point prompt and keep the box prompt.
        batched_input["point_coords"] = None
        flag = "boxes"

        # Freeze the encoder backbone and train only the adapters
        for n, value in model.image_encoder.named_parameters():
            if "Adapter" in n:
                value.requires_grad = True
            else:
                value.requires_grad = False

        labels = batched_input["label"]
        image_embeddings = model.image_encoder(batched_input["image"])

        B, _, _, _ = image_embeddings.shape
        image_embeddings_repeat = []
        for i in range(B):
            image_embed = image_embeddings[i].repeat(args.mask_num, 1, 1, 1)
            image_embeddings_repeat.append(image_embed)
        image_embeddings = torch.cat(image_embeddings_repeat, dim=0)

        masks, low_res_masks, iou_predictions = prompt_and_decoder(
            args, batched_input, model, image_embeddings, decoder_iter=False)
        loss = criterion(masks, labels, iou_predictions)
        loss.backward(retain_graph=False)
        optimizer.step()
        optimizer.zero_grad()

        if int(batch + 1) % 50 == 0:
            print(f'Epoch: {epoch+1}, Batch: {batch+1}, first {flag} prompt: {SegMetrics(masks, labels, args.metrics)}')

        # --- Refinement loop (original implementation: iter_point iterations) ---
        point_num = random.choice(args.point_list)
        batched_input = generate_point(masks, labels, low_res_masks, batched_input, point_num)
        batched_input = to_device(batched_input, args.device)

        image_embeddings = image_embeddings.detach().clone()
        for n, value in model.named_parameters():
            if "image_encoder" in n:
                value.requires_grad = False
            else:
                value.requires_grad = True

        # FIX 2: prevent randint crashes when iter_point < 3 by guarding the lower bound
        init_mask_num = np.random.randint(1, max(2, args.iter_point - 1))
        for iter in range(args.iter_point):
            if iter == init_mask_num or iter == args.iter_point - 1:
                batched_input = setting_prompt_none(batched_input)

            masks, low_res_masks, iou_predictions = prompt_and_decoder(
                args, batched_input, model, image_embeddings, decoder_iter=True)
            loss = criterion(masks, labels, iou_predictions)
            loss.backward(retain_graph=True)
            optimizer.step()
            optimizer.zero_grad()

            if iter != args.iter_point - 1:
                point_num = random.choice(args.point_list)
                batched_input = generate_point(masks, labels, low_res_masks, batched_input, point_num)
                batched_input = to_device(batched_input, args.device)

            if int(batch + 1) % 50 == 0:
                if iter == init_mask_num or iter == args.iter_point - 1:
                    print(f'Epoch: {epoch+1}, Batch: {batch+1}, mask prompt: {SegMetrics(masks, labels, args.metrics)}')
                else:
                    print(f'Epoch: {epoch+1}, Batch: {batch+1}, point {point_num} prompt: {SegMetrics(masks, labels, args.metrics)}')

        if int(batch + 1) % 200 == 0:
            print(f"epoch:{epoch+1}, iteration:{batch+1}, loss:{loss.item()}")

        train_losses.append(loss.item())
        train_loader.set_postfix(train_loss=loss.item())

        train_batch_metrics = SegMetrics(masks, labels, args.metrics)
        train_iter_metrics = [train_iter_metrics[i] + train_batch_metrics[i]
                               for i in range(len(args.metrics))]

    return train_losses, train_iter_metrics



def val_one_epoch(args, model, val_loader):
    """Single-pass box inference on the validation set - matched to test.py, evaluated at 256x256."""
    model.eval()
    val_iter_metrics = [0] * len(args.metrics)
    l = len(val_loader)
    for batched_input in val_loader:
        batched_input = to_device(batched_input, args.device)
        with torch.no_grad():
            image_embeddings = model.image_encoder(batched_input["image"])
            batched_input["point_coords"] = None
            batched_input["point_labels"] = None
            masks, _, iou_predictions = prompt_and_decoder(
                args, batched_input, model, image_embeddings, decoder_iter=False)
        # Evaluate at image_size (256), consistent with the model processing resolution
        ori = F.interpolate(batched_input["ori_label"].float(),
                            (args.image_size, args.image_size), mode='nearest')
        bm = SegMetrics(masks, ori, args.metrics)
        for j in range(len(args.metrics)):
            val_iter_metrics[j] += float(bm[j])
    return [m / l for m in val_iter_metrics]

def main(args):
    model = sam_model_registry[args.model_type](args).to(args.device)
    optimizer = optim.Adam(model.parameters(), lr=args.lr)
    criterion = FocalDiceloss_IoULoss()

    if args.lr_scheduler:
        scheduler = torch.optim.lr_scheduler.MultiStepLR(optimizer, milestones=[5, 10], gamma=0.5)
        print('*******Use MultiStepLR')

    if args.resume is not None:
        with open(args.resume, "rb") as f:
            checkpoint = torch.load(f, weights_only=False)
            model.load_state_dict(checkpoint['model'])
            optimizer.load_state_dict(checkpoint['optimizer'].state_dict())
            print(f"*******load {args.resume}")

    print('*******Do not use mixed precision')

    train_dataset = TrainingDataset(args.data_path, image_size=args.image_size,
                                    mode='train', point_num=1, mask_num=args.mask_num,
                                    requires_name=False)
    train_loader = DataLoader(train_dataset, batch_size=args.batch_size,
                               shuffle=True, num_workers=2)
    print('*******Train data:', len(train_dataset))

    # Validation set (single-pass bbox, mode zoom_out)
    val_data = args.val_data_path or args.data_path
    val_dataset = TestingDataset(val_data, image_size=args.image_size,
                                 mode='val', requires_name=False,
                                 return_ori_mask=True, prompt_mode='zoom_out')
    val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False, num_workers=2)
    print('*******Val data:', len(val_dataset))

    loggers = get_logger(os.path.join(args.work_dir, "logs",
                         f"{args.run_name}_{datetime.datetime.now().strftime('%Y%m%d-%H%M.log')}"))

    best_loss = 1e10
    best_val_dice = -1.0
    no_improve = 0
    early_stop = getattr(args, 'early_stop', 15)
    l = len(train_loader)

    for epoch in range(0, args.epochs):
        model.train()
        start = time.time()
        os.makedirs(os.path.join(f"{args.work_dir}/models", args.run_name), exist_ok=True)
        train_losses, train_iter_metrics = train_one_epoch(
            args, model, optimizer, train_loader, epoch, criterion)

        if args.lr_scheduler is not None:
            scheduler.step()

        train_iter_metrics = [metric / l for metric in train_iter_metrics]
        train_metrics = {args.metrics[i]: '{:.4f}'.format(train_iter_metrics[i])
                         for i in range(len(train_iter_metrics))}

        average_loss = np.mean(train_losses)
        lr = scheduler.get_last_lr()[0] if args.lr_scheduler is not None else args.lr
        loggers.info(f"epoch: {epoch + 1}, lr: {lr}, Train loss: {average_loss:.4f}, metrics: {train_metrics}")

        # Validation: single-pass box prompts (matched to test.py), used to select the best checkpoint
        val_metrics = val_one_epoch(args, model, val_loader)
        val_dice = val_metrics[args.metrics.index('dice')] if 'dice' in args.metrics else val_metrics[0]
        val_metrics_str = {args.metrics[i]: f"{val_metrics[i]:.4f}" for i in range(len(args.metrics))}
        loggers.info(f"  → Val (single-pass bbox): {val_metrics_str}")
        print(f"  → Val Dice={val_dice:.4f} | Best={best_val_dice:.4f}")

        if val_dice > best_val_dice:
            best_val_dice = val_dice
            best_loss = average_loss  # track for reference
            no_improve = 0
            save_path = os.path.join(args.work_dir, "models", args.run_name, "best_sam.pth")
            state = {'model': model.float().state_dict(), 'optimizer': optimizer}
            torch.save(state, save_path)
            print(f"[Best] epoch {epoch+1}, val_dice={val_dice:.4f} → saved best_sam.pth")
        else:
            no_improve += 1

        end = time.time()
        print("Run epoch time: %.2fs" % (end - start))

        if no_improve >= early_stop:
            loggers.info(f"Early stopping at epoch {epoch+1} (no improve for {early_stop} epochs)")
            break

    # Save the final-epoch weights (last)
    save_path = os.path.join(args.work_dir, "models", args.run_name, "last_sam.pth")
    state = {'model': model.float().state_dict(), 'optimizer': optimizer}
    torch.save(state, save_path)
    print(f"[Last] saved last_sam.pth")


if __name__ == '__main__':
    args = parse_args()
    main(args)

Overwriting /kaggle/working/SAM-Med2D/train.py


In [ ]:
%%writefile $BASE/SAM-Med2D/segment_anything/build_sam.py
# Copyright (c) Meta Platforms, Inc. and affiliates.
# All rights reserved.

# This source code is licensed under the license found in the
# LICENSE file in the root directory of this source tree.

import torch
from functools import partial
from .modeling import ImageEncoderViT, MaskDecoder, PromptEncoder, Sam, TwoWayTransformer
from torch.nn import functional as F

def build_sam_vit_h(args):
    return _build_sam(
        encoder_embed_dim=1280,
        encoder_depth=32,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[7, 15, 23, 31],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


build_sam = build_sam_vit_h


def build_sam_vit_l(args):
    return _build_sam(
        encoder_embed_dim=1024,
        encoder_depth=24,
        encoder_num_heads=16,
        encoder_global_attn_indexes=[5, 11, 17, 23],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,
    )


def build_sam_vit_b(args):
    return _build_sam(
        encoder_embed_dim=768,
        encoder_depth=12,
        encoder_num_heads=12,
        encoder_global_attn_indexes=[2, 5, 8, 11],
        image_size=args.image_size,
        checkpoint=args.sam_checkpoint,
        encoder_adapter = args.encoder_adapter,

    )


sam_model_registry = {
    "default": build_sam_vit_h,
    "vit_h": build_sam_vit_h,
    "vit_l": build_sam_vit_l,
    "vit_b": build_sam_vit_b,
}


def _build_sam(
    encoder_embed_dim,
    encoder_depth,
    encoder_num_heads,
    encoder_global_attn_indexes,
    image_size,
    checkpoint,
    encoder_adapter,
):
    prompt_embed_dim = 256
    image_size = image_size
    vit_patch_size = 16
    image_embedding_size = image_size // vit_patch_size
    sam = Sam(
        image_encoder=ImageEncoderViT(
            depth=encoder_depth,
            embed_dim=encoder_embed_dim,
            img_size=image_size,
            mlp_ratio=4,
            norm_layer=partial(torch.nn.LayerNorm, eps=1e-6),
            num_heads=encoder_num_heads,
            patch_size=vit_patch_size,
            qkv_bias=True,
            use_rel_pos = True,
            global_attn_indexes=encoder_global_attn_indexes,
            window_size=14,
            out_chans=prompt_embed_dim,
            adapter_train = encoder_adapter,
        ),
        prompt_encoder=PromptEncoder(
            embed_dim=prompt_embed_dim,
            image_embedding_size=(image_embedding_size, image_embedding_size),
            input_image_size=(image_size, image_size),
            mask_in_chans=16,
        ),
        mask_decoder=MaskDecoder(
            num_multimask_outputs=3,
            transformer=TwoWayTransformer(
                depth=2,
                embedding_dim=prompt_embed_dim,
                mlp_dim=2048,
                num_heads=8,
            ),
            transformer_dim=prompt_embed_dim,
            iou_head_depth=3,
            iou_head_hidden_dim=256,
        ),
        pixel_mean=[123.675, 116.28, 103.53],
        pixel_std=[58.395, 57.12, 57.375],
    )
    # sam.train()
    if checkpoint is not None:
        with open(checkpoint, "rb") as f:
            # Force weights_only=False so that newer PyTorch versions can read the checkpoint containing the authors' Adam optimizer state
            state_dict = torch.load(f, map_location="cpu", weights_only=False)
        try:
            if 'model' in state_dict.keys():
                print(encoder_adapter)
                sam.load_state_dict(state_dict['model'], False) # The original implementation already uses strict=False here
            else:
                if image_size==1024 and encoder_adapter==True:
                    sam.load_state_dict(state_dict, False)
                else:
                    sam.load_state_dict(state_dict)
        except:
            print('*******interpolate')
            new_state_dict = load_from(sam, state_dict, image_size, vit_patch_size)
            sam.load_state_dict(new_state_dict)
        print(f"*******load {checkpoint}")

    return sam


def load_from(sam, state_dicts, image_size, vit_patch_size):

    sam_dict = sam.state_dict()
    except_keys = ['mask_tokens', 'output_hypernetworks_mlps', 'iou_prediction_head']
    new_state_dict = {k: v for k, v in state_dicts.items() if
                      k in sam_dict.keys() and except_keys[0] not in k and except_keys[1] not in k and except_keys[2] not in k}
    pos_embed = new_state_dict['image_encoder.pos_embed']
    token_size = int(image_size // vit_patch_size)
    if pos_embed.shape[1] != token_size:
        # resize pos embedding, which may sacrifice the performance, but I have no better idea
        pos_embed = pos_embed.permute(0, 3, 1, 2)  # [b, c, h, w]
        pos_embed = F.interpolate(pos_embed, (token_size, token_size), mode='bilinear', align_corners=False)
        pos_embed = pos_embed.permute(0, 2, 3, 1)  # [b, h, w, c]
        new_state_dict['image_encoder.pos_embed'] = pos_embed
        rel_pos_keys = [k for k in sam_dict.keys() if 'rel_pos' in k]

        global_rel_pos_keys = [k for k in rel_pos_keys if
                                                        '2' in k or
                                                        '5' in k or
                                                        '7' in k or
                                                        '8' in k or
                                                        '11' in k or
                                                        '13' in k or
                                                        '15' in k or
                                                        '23' in k or
                                                        '31' in k]
        # print(sam_dict)
        for k in global_rel_pos_keys:
            h_check, w_check = sam_dict[k].shape
            rel_pos_params = new_state_dict[k]
            h, w = rel_pos_params.shape
            rel_pos_params = rel_pos_params.unsqueeze(0).unsqueeze(0)
            if h != h_check or w != w_check:
                rel_pos_params = F.interpolate(rel_pos_params, (h_check, w_check), mode='bilinear', align_corners=False)

            new_state_dict[k] = rel_pos_params[0, 0, ...]

    sam_dict.update(new_state_dict)
    return sam_dict

Overwriting /kaggle/working/SAM-Med2D/segment_anything/build_sam.py


In [ ]:
%%writefile $BASE/SAM-Med2D/DataLoader.py
# (Write this before training; TrainingDataset now uses the same PGA covering-prompt
# family as TestingDataset (zoom-out + shift, 50/50 per sample), so train and val/test
# share one prompt distribution)
import os
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2
import cv2
import torch
import numpy as np
from torch.nn import functional as F
from torch.utils.data import DataLoader
from tqdm import tqdm
from utils import train_transforms, init_point_sampling
import json
import random

class TestingDataset(Dataset):
    def __init__(self, data_path, image_size=256, mode='test', requires_name=True, point_num=1, return_ori_mask=True, prompt_path=None, prompt_mode='zoom_out', zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        """
        Initializes a TestingDataset object.
        """
        self.image_size = image_size
        self.return_ori_mask = return_ori_mask
        self.prompt_path = prompt_path
        self.prompt_list = {} if prompt_path is None else json.load(open(prompt_path, "r"))
        self.requires_name = requires_name
        self.point_num = point_num
        self.mode = mode
        self.is_train = (mode == 'train')

        # Add variables controlling robust box perturbation
        self.prompt_mode = prompt_mode
        self.zoom_ratio = zoom_ratio
        self.shift_ratio = shift_ratio

        json_file = open(os.path.join(data_path, f'label2image_{mode}.json'), "r")
        dataset = json.load(json_file)

        sorted_items = sorted(dataset.items(), key=lambda x: os.path.basename(x[0]))
        self.label_paths = [k for k, v in sorted_items]
        self.image_paths = [v for k, v in sorted_items]

        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

    def _zoom_out_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        lo, hi = self.zoom_ratio
        if self.is_train:
            r_l, r_r = random.uniform(lo, hi), random.uniform(lo, hi)
            r_t, r_b = random.uniform(lo, hi), random.uniform(lo, hi)
        else:
            r_l = r_r = r_t = r_b = 0.30
        bx_min = max(0,       x_min - gt_w * r_l)
        bx_max = min(orig_w,  x_max + gt_w * r_r)
        by_min = max(0,       y_min - gt_h * r_t)
        by_max = min(orig_h,  y_max + gt_h * r_b)
        return bx_min, bx_max, by_min, by_max

    def _shift_bbox(self, x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=None):
        gt_w, gt_h = x_max - x_min, y_max - y_min
        bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)

        left_margin   = max(0.0, x_min - bx_min)
        right_margin  = max(0.0, bx_max - x_max)
        top_margin    = max(0.0, y_min - by_min)
        bottom_margin = max(0.0, by_max - y_max)

        if self.is_train:
            dx = random.uniform(-right_margin * self.shift_ratio, left_margin * self.shift_ratio)
            dy = random.uniform(-bottom_margin * self.shift_ratio, top_margin * self.shift_ratio)
        else:
            rng = random.Random(seed_idx or 0)
            dx = left_margin * self.shift_ratio if rng.random() < 0.5 else -right_margin * self.shift_ratio
            dy = top_margin * self.shift_ratio if rng.random() < 0.5 else -bottom_margin * self.shift_ratio

        bx_min = max(0,       bx_min + dx)
        bx_max = min(orig_w,  bx_max + dx)
        by_min = max(0,       by_min + dy)
        by_max = min(orig_h,  by_max + dy)
        return bx_min, bx_max, by_min, by_max

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        mask_path = self.label_paths[index]
        ori_np_mask = cv2.imread(mask_path, 0)

        if ori_np_mask.max() == 255:
            ori_np_mask = ori_np_mask / 255

        assert np.array_equal(ori_np_mask, ori_np_mask.astype(bool)), f"Mask should only contain binary values 0 and 1. {self.label_paths[index]}"

        h, w = ori_np_mask.shape
        ori_mask = torch.tensor(ori_np_mask).unsqueeze(0)

        transforms = train_transforms(self.image_size, h, w)
        augments = transforms(image=image, mask=ori_np_mask)
        image, mask = augments['image'], augments['mask'].to(torch.int64)

        if self.prompt_path is None:
            # Extract a tight box from the transformed mask
            y_indices, x_indices = torch.where(mask > 0)
            if len(y_indices) > 0 and len(x_indices) > 0:
                y_min, y_max = y_indices.min().item(), y_indices.max().item()
                x_min, x_max = x_indices.min().item(), x_indices.max().item()
            else:
                x_min, x_max, y_min, y_max = 0, self.image_size, 0, self.image_size

            orig_h, orig_w = self.image_size, self.image_size

            # Apply bounding-box perturbation scenarios
            if self.prompt_mode == 'zoom_out':
                bx_min, bx_max, by_min, by_max = self._zoom_out_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w)
            elif self.prompt_mode == 'shift':
                bx_min, bx_max, by_min, by_max = self._shift_bbox(x_min, x_max, y_min, y_max, orig_h, orig_w, seed_idx=index)
            else:
                bx_min, bx_max, by_min, by_max = x_min, x_max, y_min, y_max

            boxes = torch.tensor([[bx_min, by_min, bx_max, by_max]], dtype=torch.float)
            point_coords, point_labels = init_point_sampling(mask, self.point_num)
        else:
            prompt_key = mask_path.split('/')[-1]
            boxes = torch.as_tensor(self.prompt_list[prompt_key]["boxes"], dtype=torch.float)
            point_coords = torch.as_tensor(self.prompt_list[prompt_key]["point_coords"], dtype=torch.float)
            point_labels = torch.as_tensor(self.prompt_list[prompt_key]["point_labels"], dtype=torch.int)

        image_input["image"] = image
        image_input["label"] = mask.unsqueeze(0)
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels
        image_input["boxes"] = boxes
        image_input["original_size"] = (h, w)
        image_input["label_path"] = '/'.join(mask_path.split('/')[:-1])

        if self.return_ori_mask:
            image_input["ori_label"] = ori_mask

        image_name = self.label_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.label_paths)

def _tight_bbox_from_mask(mask):
    h, w = mask.shape[-2:]
    y_indices, x_indices = torch.where(mask > 0)
    if len(y_indices) > 0 and len(x_indices) > 0:
        y_min, y_max = y_indices.min().item(), y_indices.max().item()
        x_min, x_max = x_indices.min().item(), x_indices.max().item()
    else:
        x_min, x_max, y_min, y_max = 0, w, 0, h
    return x_min, x_max, y_min, y_max, h, w


def _zoom_out_bbox(x_min, x_max, y_min, y_max, h, w, zoom_ratio):
    gt_w, gt_h = x_max - x_min, y_max - y_min
    lo, hi = zoom_ratio
    r_l, r_r = random.uniform(lo, hi), random.uniform(lo, hi)
    r_t, r_b = random.uniform(lo, hi), random.uniform(lo, hi)
    bx_min = max(0, x_min - gt_w * r_l)
    bx_max = min(w, x_max + gt_w * r_r)
    by_min = max(0, y_min - gt_h * r_t)
    by_max = min(h, y_max + gt_h * r_b)
    return bx_min, bx_max, by_min, by_max


def _shift_bbox(x_min, x_max, y_min, y_max, h, w, zoom_ratio, shift_ratio):
    gt_w, gt_h = x_max - x_min, y_max - y_min
    bx_min, bx_max, by_min, by_max = _zoom_out_bbox(x_min, x_max, y_min, y_max, h, w, zoom_ratio)
    dx = random.uniform(-gt_w * shift_ratio, gt_w * shift_ratio)
    dy = random.uniform(-gt_h * shift_ratio, gt_h * shift_ratio)
    bx_min = max(0, bx_min + dx)
    bx_max = min(w, bx_max + dx)
    by_min = max(0, by_min + dy)
    by_max = min(h, by_max + dy)
    # Shift mode must still cover the full GT, only changing its relative position inside the box.
    box_w = bx_max - bx_min
    box_h = by_max - by_min
    bx_min = min(bx_min, x_min)
    by_min = min(by_min, y_min)
    bx_max = max(bx_max, x_max)
    by_max = max(by_max, y_max)
    if bx_max - bx_min < box_w:
        if bx_min <= 0:
            bx_max = min(w, bx_min + box_w)
        elif bx_max >= w:
            bx_min = max(0, bx_max - box_w)
    if by_max - by_min < box_h:
        if by_min <= 0:
            by_max = min(h, by_min + box_h)
        elif by_max >= h:
            by_min = max(0, by_max - box_h)
    return bx_min, bx_max, by_min, by_max


def prompt_box_from_mask(mask, zoom_ratio, shift_ratio):
    """PGA-style training-time box: 50/50 zoom-out expansion or off-center shift,
    mirroring PromptSegmentationDataset._zoom_out_bbox/_shift_bbox's train branches."""
    x_min, x_max, y_min, y_max, h, w = _tight_bbox_from_mask(mask)
    if random.random() < 0.5:
        bx_min, bx_max, by_min, by_max = _zoom_out_bbox(x_min, x_max, y_min, y_max, h, w, zoom_ratio)
    else:
        bx_min, bx_max, by_min, by_max = _shift_bbox(x_min, x_max, y_min, y_max, h, w, zoom_ratio, shift_ratio)
    return torch.tensor([[bx_min, by_min, bx_max, by_max]], dtype=torch.float)


class TrainingDataset(Dataset):
    def __init__(self, data_dir, image_size=256, mode='train', requires_name=True, point_num=1, mask_num=5, zoom_ratio=(0.15, 0.45), shift_ratio=0.30):
        self.image_size = image_size
        self.requires_name = requires_name
        self.point_num = point_num
        self.mask_num = mask_num
        self.zoom_ratio = zoom_ratio
        self.shift_ratio = shift_ratio
        self.pixel_mean = [123.675, 116.28, 103.53]
        self.pixel_std = [58.395, 57.12, 57.375]

        dataset = json.load(open(os.path.join(data_dir, f'image2label_{mode}.json'), "r"))
        self.image_paths = list(dataset.keys())
        self.label_paths = list(dataset.values())

    def __getitem__(self, index):
        image_input = {}
        try:
            image = cv2.imread(self.image_paths[index])
            image = (image - self.pixel_mean) / self.pixel_std
        except:
            print(self.image_paths[index])

        h, w, _ = image.shape
        transforms = train_transforms(self.image_size, h, w)

        masks_list = []
        boxes_list = []
        point_coords_list, point_labels_list = [], []
        mask_path = random.choices(self.label_paths[index], k=self.mask_num)
        for m in mask_path:
            pre_mask = cv2.imread(m, 0)
            if pre_mask.max() == 255:
                pre_mask = pre_mask / 255

            augments = transforms(image=image, mask=pre_mask)
            image_tensor, mask_tensor = augments['image'], augments['mask'].to(torch.int64)

            boxes = prompt_box_from_mask(mask_tensor, self.zoom_ratio, self.shift_ratio)
            point_coords, point_label = init_point_sampling(mask_tensor, self.point_num)

            masks_list.append(mask_tensor)
            boxes_list.append(boxes)
            point_coords_list.append(point_coords)
            point_labels_list.append(point_label)

        mask = torch.stack(masks_list, dim=0)
        boxes = torch.stack(boxes_list, dim=0)
        point_coords = torch.stack(point_coords_list, dim=0)
        point_labels = torch.stack(point_labels_list, dim=0)

        image_input["image"] = image_tensor.unsqueeze(0)
        image_input["label"] = mask.unsqueeze(1)
        image_input["boxes"] = boxes
        image_input["point_coords"] = point_coords
        image_input["point_labels"] = point_labels

        image_name = self.image_paths[index].split('/')[-1]
        if self.requires_name:
            image_input["name"] = image_name
            return image_input
        else:
            return image_input

    def __len__(self):
        return len(self.image_paths)

def stack_dict_batched(batched_input):
    out_dict = {}
    for k,v in batched_input.items():
        if isinstance(v, list):
            out_dict[k] = v
        else:
            out_dict[k] = v.reshape(-1, *v.shape[2:])
    return out_dict

Overwriting /kaggle/working/SAM-Med2D/DataLoader.py


In [ ]:
# Training: PGA covering-prompt protocol, 50/50 per sample between zoom-out
# expansion (ratio 0.15-0.45) and off-center shift (ratio 0.30), instead of the
# authors' box-noise. PGA-UNet itself trains zoom-out only; SAM-Med2D trains on
# both modes here so it is not undertrained relative to what it is tested on.
# Validation-during-training and test.py use the PGA covering-prompt protocol too
# (prompt_mode zoom_out/shift, ratio 0.30)
# iter_point=0  : box-only, no point-prompt refinement iterations
# mask_num=5    : 5 mask augmentations per image (effective batch = batch_size x mask_num)
# batch_size=4  : matches PGA-UNet; effective batch 20, watch for OOM on a T4
# epochs=100, early_stop=30 (matches PGA-UNet epoch budget so patience 30 is meaningful)
%cd $BASE/SAM-Med2D
!python train.py \
    --work_dir "workdir" \
    --image_size 256 \
    --mask_num 5 \
    --data_path "dataset_BTXRD" \
    --sam_checkpoint "checkpoint/sam_med2d.pth" \
    --iter_point 0 \
    --encoder_adapter True \
    --epochs 100 \
    --early_stop 30 \
    --batch_size 4 \
    --lr 1e-5

/kaggle/working/SAM-Med2D
True
*******load checkpoint/sam_med2d.pth
*******Do not use mixed precision
*******Train data: 1493
*******Val data: 239
 53%|████████████▏          | 199/374 [02:11<01:35,  1.83it/s, train_loss=0.594]Epoch: 1, Batch: 200, first boxes prompt: [0.21337266 0.29245082]
epoch:1, iteration:200, loss:0.6683427095413208
100%|███████████████████████| 374/374 [03:59<00:00,  1.56it/s, train_loss=0.749]
[2026-08-13 17:09:36,327][train.py][line:288][INFO] epoch: 1, lr: 1e-05, Train loss: 0.5607, metrics: {'iou': '0.4957', 'dice': '0.6263'}
[2026-08-13 17:10:13,033][train.py][line:294][INFO]   → Val (single-pass bbox): {'iou': '0.5569', 'dice': '0.6821'}
  → Val Dice=0.6821 | Best=-1.0000
[Best] epoch 1, val_dice=0.6821 → saved best_sam.pth
Run epoch time: 279.13s
 53%|████████████▏          | 199/374 [02:12<01:58,  1.47it/s, train_loss=0.341]Epoch: 2, Batch: 200, first boxes prompt: [0.65675396 0.79041255]
epoch:2, iteration:200, loss:0.3824649751186371
100%|█████████████

In [ ]:
import glob, os, shutil

# Collect the best and last checkpoints
best_ckpt = f"{BASE}/SAM-Med2D/workdir/models/sam-med2d/best_sam.pth"
last_ckpt = f"{BASE}/SAM-Med2D/workdir/models/sam-med2d/last_sam.pth"

os.makedirs(f"{BASE}/drive/MyDrive/model", exist_ok=True)
for ckpt in [best_ckpt, last_ckpt]:
    if os.path.exists(ckpt):
        shutil.copy(ckpt, f"{BASE}/drive/MyDrive/model/")
        print(f"✅ Saved to Drive: {os.path.basename(ckpt)}")
    else:
        print(f"⚠️ Not found: {os.path.basename(ckpt)}")

✅ Saved to Drive: best_sam.pth
✅ Saved to Drive: last_sam.pth
